# Method 1 - Part 5: extra seeds on the five folds (TESSERA vs concatenation)

In Part 4 (one seed per fold), early concatenation beat TESSERA alone by +0.84 mIoU points (95% CI [+0.47, +1.21]) pooled over all 2,433 patches and finished ahead on all five folds. Because patch bootstrapping does not reflect training-seed variance, and fold I single-run differences varied across seeds (+1.04, -0.23, and 0.00 for seeds 0, 1, and 2), we run seeds 1 and 2 for both models on folds II to V (16 new runs under `configs/common/`). Together with fold I seeds 0 to 2 from Part 3 and folds II to V seed 0 from Part 4, we pool predictions across all five folds per seed and average pooled mIoU and paired differences over the three seeds.

| Variant | Description |
|---|---|
| `base_unet-t19-lr1e3-nocw-w32` | TESSERA 2019 alone (128 ch.) |
| `m1_unet-t19-ae19-lr1e3-nocw-w32` | Early concatenation of TESSERA 2019 + AlphaEarth 2019 (192 ch.) |

## 1. Environment and paths

In [ ]:
import json, os, shutil, platform, sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
print("Python", platform.python_version(), "| torch", torch.__version__, f"| {os.cpu_count()} CPUs")
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > T4 GPU, then run again"
print("GPU:", torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/pj_vmi")
REPO_URL = "https://github.com/ColdCastlee/ProjectVMI.git"
BRANCH = "main"  # or the branch being tested
CODE = Path("/content/code")
PROC_DRIVE = DRIVE / "processed"
PROC = Path("/content/processed")
# runs and results; an account that reaches pj_vmi through a shortcut to a folder shared by another
# account cannot create folders in it: set WORK to a folder of its own, e.g. Path("/content/drive/MyDrive/pj_vmi_runs_alt")
WORK = DRIVE
RUNS, RUNS_CSV = WORK / "runs", WORK / "results" / "runs.csv"
OUT = WORK / "results" / "m1"
CONFIGS = ["configs/common/b2_unet_t19.yaml", "configs/common/m1_1_concat_t19_ae19.yaml"]
T, CONCAT = "base_unet-t19-lr1e3-nocw-w32", "m1_unet-t19-ae19-lr1e3-nocw-w32"
NEW_SEEDS = "1 2"

need = [(v, "I", s) for v in (T, CONCAT) for s in (0, 1, 2)]
need += [(v, f, 0) for v in (T, CONCAT) for f in ("II", "III", "IV", "V")]
missing = [f"{v}_100_fold{f}_s{s}" for v, f, s in need if not (RUNS / f"{v}_100_fold{f}_s{s}" / "run.json").exists()]
assert not missing, f"missing runs {missing}: run notebooks/m1/03_fusion_unet.ipynb and 04_five_folds.ipynb first"
print("fold I seeds 0, 1, 2 and folds II-V seed 0 found for TESSERA alone and the concatenation")

## 2. Repository and unit tests

In [ ]:
if (CODE / ".git").exists():
    !git -C "{CODE}" fetch -q origin {BRANCH} && git -C "{CODE}" checkout -q {BRANCH} && git -C "{CODE}" pull -q --ff-only origin {BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} "{CODE}"
%cd {CODE}
!git log -1 --format="commit %h  %ad  %s" --date=short
!pip install -q -r requirements.txt
!python -m pytest -q tests
sys.path.insert(0, str(CODE))

## 3. Copy TESSERA 2019 and AlphaEarth 2019 to local disk

In [ ]:
PROC.mkdir(parents=True, exist_ok=True)
for f in ["meta.csv", "splits.json", "labels.npy", "instances.npy", "valid_mask.npy",
          "tessera_2019_q.npy", "tessera_2019_scale.npy", "ae_2019.npy"]:
    if not (PROC / f).exists() or (PROC / f).stat().st_size != (PROC_DRIVE / f).stat().st_size:
        shutil.copy(PROC_DRIVE / f, PROC / f)
    print(f"{f:24s} {(PROC / f).stat().st_size / 1e9:6.2f} GB  ok={(PROC / f).stat().st_size == (PROC_DRIVE / f).stat().st_size}")

## 4. Fold II
Train TESSERA alone and concatenation for seeds 1 and 2 (seed 0 comes from Part 4).

In [ ]:
SPLIT = "II"
for cfg in CONFIGS:
    !python scripts/train_head.py --config {cfg} --processed {PROC} --runs-dir {RUNS} --runs-csv {RUNS_CSV} --split {SPLIT} --seeds {NEW_SEEDS}

## 5. Fold III

In [ ]:
SPLIT = "III"
for cfg in CONFIGS:
    !python scripts/train_head.py --config {cfg} --processed {PROC} --runs-dir {RUNS} --runs-csv {RUNS_CSV} --split {SPLIT} --seeds {NEW_SEEDS}

## 6. Fold IV

In [ ]:
SPLIT = "IV"
for cfg in CONFIGS:
    !python scripts/train_head.py --config {cfg} --processed {PROC} --runs-dir {RUNS} --runs-csv {RUNS_CSV} --split {SPLIT} --seeds {NEW_SEEDS}

## 7. Fold V

In [ ]:
SPLIT = "V"
for cfg in CONFIGS:
    !python scripts/train_head.py --config {cfg} --processed {PROC} --runs-dir {RUNS} --runs-csv {RUNS_CSV} --split {SPLIT} --seeds {NEW_SEEDS}

## 8. Five folds, three seeds
Pools per-patch confusion matrices across folds I to V (all 2,433 patches) per seed, then averages pooled mIoU and paired differences over seeds 0, 1, and 2 alongside the per-run breakdown across all 15 `(fold, seed)` pairs.

In [ ]:
from IPython.display import Image, display
from scripts.summarize_folds import summarize_folds

pd.set_option("display.width", 220)
per_fold, summary, deltas = summarize_folds(RUNS, OUT, name="fusion_unet_5folds_3seeds-lr1e3-nocw", variants=[T, CONCAT],
                                            pairs=[f"{CONCAT}:{T}"], seeds=(0, 1, 2))
runs = per_fold.pivot_table(index=["split", "seed"], columns="variant", values="miou")
runs["delta"] = runs[CONCAT] - runs[T]
runs = runs.reindex(["I", "II", "III", "IV", "V"], level="split")
print("test mIoU of every run and difference concatenation - TESSERA alone:")
print(runs.round(4).to_string())
print(f"\nsingle runs with a positive difference: {(runs.delta > 0).sum()} of {len(runs)}; "
      f"mean {runs.delta.mean():+.4f}, std {runs.delta.std(ddof=1):.4f}")
print("\npooled over the five folds, mean of the three seeds:")
print(summary[["variant", "seeds", "patches", "miou_pooled", "miou_ci_low", "miou_ci_high", "miou_std_seeds",
               "boundary_pooled", "core_pooled", "miou_fold_mean", "miou_fold_std"]].round(4).to_string(index=False))
print("\npaired bootstrap on the pooled patches, delta = mIoU(variant) - mIoU(versus), mean of the three seeds:")
print(deltas.round(4).to_string(index=False))
display(Image(str(OUT / "fusion_unet_5folds_3seeds-lr1e3-nocw_per_class.png")))